In [1]:
import pandas as pd 
import pyarrow
import duckdb
import os
import requests
from datetime import datetime
from dotenv import load_dotenv , find_dotenv


# 1. 환경 변수 로드
load_dotenv(find_dotenv())
OPINET_API_KEY_GROUP = ['OPINET_API_KEY_1', 'OPINET_API_KEY_2', 'OPINET_API_KEY_3', 
                        'OPINET_API_KEY_4', 'OPINET_API_KEY_5', 'OPINET_API_KEY_6' ] # OPINET_API_KEY가 여러개로 늘어날 수도 있으니까 list로 관리
# API_KEY = os.getenv('OPINET_API_KEY_1') --> 이건 이제 동적변수가 되어야 하므로 함수 안으로 집어넣기
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT') 
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')


#2. duckdb를 통한 s3 읽기 설정
duckdb.execute("INSTALL httpfs; LOAD httpfs;")
duckdb.execute(f"SET s3_endpoint='{MINIO_ENDPOINT}';")
duckdb.execute(f"SET s3_access_key_id='{MINIO_ACCESS_KEY}';")
duckdb.execute(f"SET s3_secret_access_key='{MINIO_SECRET_KEY}';")
duckdb.execute("SET s3_url_style='path'; SET s3_use_ssl='false';")

#3. API 기본 호출 URL
url = "https://www.opinet.co.kr/api/aroundAll.do"

# 일배치를 통해 동적으로 변화하는 마스터 테이블의 s3 경로
target_coordinates_path = "s3://petroleum-project/target_coordinates/target_coordinates.parquet"
target_coordinates = duckdb.read_parquet(target_coordinates_path).df()

In [2]:
target_coordinates[target_coordinates['is_gasoline_check']==1]

,katec_x,katec_y,lat,lon,is_gasoline_check,is_premium_gasoline_check,is_lpg_check,last_check_gasoline,last_check_premium_gasoline,last_check_lpg
0,302130.89,552206.98,37.564114,126.892029,1,1,1,20260203,20260206,20260208
1,285130.89,537484.55,37.429508,126.701913,1,1,1,20260203,20260206,20260208
2,310630.89,581651.84,37.830283,126.984630,1,1,0,20260203,20260206,None
3,365880.89,500678.47,37.104308,127.616081,1,0,1,20260203,None,20260208
4,327630.89,581651.84,37.831789,127.177756,1,1,0,20260203,20260206,None
...,...,...,...,...,...,...,...,...,...,...
1442,306380.89,309286.85,35.375364,126.969532,1,0,0,20260202,None,None
1443,370130.89,625819.14,38.232160,127.658778,1,0,0,20260204,None,None
1446,421130.89,316648.07,35.445895,128.232795,1,0,0,20260202,None,None
1447,229880.89,206229.83,34.436599,126.148769,1,0,0,20260202,None,None


In [4]:
target_df_filtered = target_coordinates[target_coordinates['is_gasoline_check'] == 1 ]
display(target_df_filtered)

,katec_x,katec_y,lat,lon,is_gasoline_check,is_premium_gasoline_check,is_lpg_check,last_check_gasoline,last_check_premium_gasoline,last_check_lpg
0,302130.89,552206.98,37.564114,126.892029,1,1,1,20260203,20260206,20260208
1,285130.89,537484.55,37.429508,126.701913,1,1,1,20260203,20260206,20260208
2,310630.89,581651.84,37.830283,126.984630,1,1,0,20260203,20260206,None
3,365880.89,500678.47,37.104308,127.616081,1,0,1,20260203,None,20260208
4,327630.89,581651.84,37.831789,127.177756,1,1,0,20260203,20260206,None
...,...,...,...,...,...,...,...,...,...,...
1442,306380.89,309286.85,35.375364,126.969532,1,0,0,20260202,None,None
1443,370130.89,625819.14,38.232160,127.658778,1,0,0,20260204,None,None
1446,421130.89,316648.07,35.445895,128.232795,1,0,0,20260202,None,None
1447,229880.89,206229.83,34.436599,126.148769,1,0,0,20260202,None,None


In [10]:
params = {
                    "code": os.getenv(OPINET_API_KEY_GROUP[5]),
                    "out": "json",
                    "x": "433880.89",
                    "y": "441788.74",
                    "radius": 5000,
                    "prodcd": "B027" , #"B027",  # 휘발유 기준 (경유는 D047)
                    "sort": 1          # 1: 가격순, 2: 거리순
                    }
response = requests.get(url, params=params)
response.raise_for_status() # 에러 발생 시 예외 처리
data = response.json()

display(response) 
display(data)

<Response [200]>

{'RESULT': {'OIL': []}}

In [12]:
api_check = data.get('RESULT',{}).get('OIL')
display(api_check)

[]

In [13]:
params = {
                    "code": os.getenv(OPINET_API_KEY_GROUP[0]),
                    "out": "json",
                    "x": "433880.89",
                    "y": "441788.74",
                    "radius": 5000,
                    "prodcd": "B027" , #"B027",  # 휘발유 기준 (경유는 D047)
                    "sort": 1          # 1: 가격순, 2: 거리순
                    }
response = requests.get(url, params=params)
response.raise_for_status() # 에러 발생 시 예외 처리
data_0 = response.json()

display(response) 
display(data_0)

<Response [200]>

{'RESULT': {'OIL': [{'UNI_ID': 'A0025539',
    'POLL_DIV_CD': 'GSC',
    'OS_NM': '착한셀프주유소',
    'PRICE': 1675,
    'DISTANCE': 4943.1,
    'GIS_X_COOR': 433856.0,
    'GIS_Y_COOR': 446732.0},
   {'UNI_ID': 'A0025880',
    'POLL_DIV_CD': 'SKE',
    'OS_NM': '지보농협주유소',
    'PRICE': 1715,
    'DISTANCE': 3690.0,
    'GIS_X_COOR': 434852.0,
    'GIS_Y_COOR': 438228.0},
   {'UNI_ID': 'A0025813',
    'POLL_DIV_CD': 'SOL',
    'OS_NM': '송평주유소',
    'PRICE': 1719,
    'DISTANCE': 2833.2,
    'GIS_X_COOR': 436548.0,
    'GIS_Y_COOR': 440832.0}]}}

In [14]:
api_check_0 = data_0.get('RESULT',{}).get('OIL')
display(api_check_0)

[{'UNI_ID': 'A0025539',
  'POLL_DIV_CD': 'GSC',
  'OS_NM': '착한셀프주유소',
  'PRICE': 1675,
  'DISTANCE': 4943.1,
  'GIS_X_COOR': 433856.0,
  'GIS_Y_COOR': 446732.0},
 {'UNI_ID': 'A0025880',
  'POLL_DIV_CD': 'SKE',
  'OS_NM': '지보농협주유소',
  'PRICE': 1715,
  'DISTANCE': 3690.0,
  'GIS_X_COOR': 434852.0,
  'GIS_Y_COOR': 438228.0},
 {'UNI_ID': 'A0025813',
  'POLL_DIV_CD': 'SOL',
  'OS_NM': '송평주유소',
  'PRICE': 1719,
  'DISTANCE': 2833.2,
  'GIS_X_COOR': 436548.0,
  'GIS_Y_COOR': 440832.0}]